In [0]:
# COMMAND ----------
# CELL 1: Setup and Data Ingestion
import pandas as pd
import requests
import io
from pyspark.sql.functions import col, upper, trim, lpad, when

# Source IDs from your Colab notebook
SNF_ID = "1UfCxgMxUtCEDWqcm1udnd7mPawDh7y-b"
PBJ_ID = "1y9WofLddBZ7ufuAeJ0HEfW9uRlvuQTt7"
ADM_ID = "1mR7vOR3xyeZ6sv4QiclCftOYqB79bajT"

def download_to_spark(file_id):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    res.raise_for_status()
    pdf = pd.read_csv(io.BytesIO(res.content), low_memory=False)
    pdf.columns = [str(c).strip() for c in pdf.columns]
    return spark.createDataFrame(pdf)

try:
    print("Downloading and processing files...")
    
    # 1. Facilities DataFrame
    facilities_df = download_to_spark(SNF_ID).select(
        upper(trim(col("CMS Certification Number (CCN)"))).alias("cms_id"),
        col("Provider Name").alias("name"),
        col("Chain Name").alias("chain_name")
    )
    facilities_df.createOrReplaceTempView("facilities")

    # 2. PBJ Staffing (Creating pbj_long_df)
    # We standardize to 6-digit CMS ID and unpivot the hours
    raw_pbj = download_to_spark(PBJ_ID)
    pbj_long_df = raw_pbj.withColumn("cms_id", lpad(upper(trim(col("PROVNUM"))), 6, "0")) \
        .selectExpr("cms_id", "stack(3, 'RN', HRS_RN_CTR, 'LPN', HRS_LPN_CTR, 'CNA', HRS_CNA_CTR) as (job_type, total_hours)") \
        .filter("total_hours > 0")
    pbj_long_df.createOrReplaceTempView("pbj_hours")

    # 3. Admin Details
    admin_df = download_to_spark(ADM_ID)
    admin_df.createOrReplaceTempView("admin_details")

    print("✅ SUCCESS: All DataFrames and SQL Views are ready.")
except Exception as e:
    print(f"❌ Setup Failed: {e}")

In [0]:
# COMMAND ----------
# CELL 2: Top 10 Chains by Agency Staffing Volume

from pyspark.sql.functions import sum as _sum, round

# Calculate the total hours across the whole state for percentage context
total_state_hours = pbj_long_df.select(_sum("total_hours")).collect()[0][0]

# Aggregate by chain_name
chain_report = facilities_df.join(pbj_long_df, "cms_id") \
    .groupBy("chain_name") \
    .agg(_sum("total_hours").alias("total_agency_hours")) \
    .withColumn("pct_of_state_total", round((col("total_agency_hours") / total_state_hours) * 100, 2)) \
    .filter(col("chain_name").isNotNull()) \
    .orderBy(col("total_agency_hours").desc()) \
    .limit(10)

print("📊 Top 10 Chains by Staffing Volume:")
display(chain_report)

In [0]:
# COMMAND ----------
# CELL 3: Top 100 Facilities - Administrator Contact List

from pyspark.sql.functions import sum as _sum, col, upper, trim

# 1. Aggregate hours to find Top 100
top_100_stats = pbj_long_df.groupBy("cms_id") \
    .agg(_sum("total_hours").alias("total_hours")) \
    .orderBy(col("total_hours").desc()) \
    .limit(100).alias("t")

# 2. Alias tables to avoid "Ambiguous Column" errors
f = facilities_df.alias("f")
a = admin_df.alias("a")

# 3. Final Join and Selection
contact_list = top_100_stats.join(f, col("t.cms_id") == col("f.cms_id")) \
    .join(a, (upper(trim(col("f.name"))) == upper(trim(col("a.FACNAME")))), "left") \
    .select(
        col("f.cms_id"), 
        col("f.name").alias("facility_name"), 
        col("a.FACADMIN").alias("admin_name"), 
        col("a.CONTACT_EMAIL").alias("admin_email"),
        col("t.total_hours")
    )

print("✅ Contact list for Top 100 facilities generated.")
display(contact_list)